# Marketing leads prediction

Synthetic, executed project walkthrough. This notebook can reproduce the delivered run. Start Jupyter in the extracted project directory. The cells below are unexecuted; the bundled outputs and reported metrics were produced by run_pipeline.py.

## 1. Define the target and data contract
Predict next week’s total inquiries from planned budgets, prior-week observations and known calendar signals. Read README.md for availability rules.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
from run_pipeline import generate_data, clean_data, make_features, run
raw, future = generate_data()
clean, quality = clean_data(raw)
display(pd.Series(quality))
display(clean.head())

## 2. Feature engineering
All realized outcomes and engagement features are shifted. Current-week plans and calendar are known in advance.

In [ ]:
features = make_features(clean)
display(features.iloc[52:].head())
print(features.columns.tolist())

## 3. Train, tune and evaluate
52 weeks of warm-up, 210 training, 26 calibration, 24 holdout. Four expanding CV folds. Winner chosen before final holdout. This reruns all model fits.

In [ ]:
summary = run(out='outputs')
display(pd.read_csv('outputs/model_comparison.csv'))
display(pd.read_csv('outputs/cv_folds.csv'))

## 4. Inspect errors and model reliance

In [ ]:
display(Image(filename='outputs/results_dashboard.png'))
display(pd.read_csv('outputs/feature_importance.csv').head(10))
print('Mean bias:', summary['mean_bias_actual_minus_prediction'])

## 5. Score from the saved artifact

In [ ]:
from score import score
score('outputs/model.joblib', 'outputs/clean_weekly_data.csv', 'outputs/future_inputs.json')

## 6. Monitoring and production handoff

In [ ]:
display(pd.read_csv('outputs/drift_snapshot.csv'))
print(Path('README.md').read_text())

## What has and has not been done
- **1. Business problem — Executed:** Forecast next week’s total inquiries for an education-style marketing funnel. Use MAE for selection; support workload planning.
- **2. Data collection — Simulated:** 312 weeks of generated CRM, website and paid-media style data. No Salesforce, GA4 or advertising account was connected.
- **3. Data integration — Executed on simulated tables:** SQLite joins of weekly CRM, ads and GA4 tables; joined data reconciled exactly to the clean input. Production source extraction remains to be connected.
- **4. Data quality — Executed:** Required schema, numeric types, nonnegative values, integer leads, weekly continuity, binary deadlines and unique dates checked.
- **5. EDA — Executed:** Training-only trend, histogram, spend scatterplot, correlation matrix and descriptive statistics.
- **6. Cleaning — Executed:** Removed 2 exact duplicates. Eight missing sessions and four missing clicks become lagged missing inputs; fold-local median imputation handles them. No target imputation or blanket outlier deletion.
- **7. Feature engineering — Executed:** 24 features: lead lags 1/2/4/13/52; shifted 4/13-week rolling means and standard deviations; lagged engagement, CTR and lead/session rate; planned spend and log-spend; trend; annual and semiannual Fourier terms; deadline indicator.
- **8. Feature selection — Executed:** Fold-local removal of zero-variance features; Ridge/Elastic Net regularization. Training correlation diagnostics retained. No post-holdout feature selection.
- **9. Data splitting — Executed:** First 52 weeks initialize lag history; then 210 training, 26 calibration and 24 holdout weeks. No shuffled split.
- **10. Baselines — Executed:** Previous week, previous four-week mean and 52-week seasonal naive.
- **11. Model development — Executed:** Ridge, Elastic Net, Random Forest and scikit-learn Histogram Gradient Boosting. XGBoost, LightGBM, CatBoost, SARIMA and neural networks were not run.
- **12. Hyperparameter tuning — Executed:** GridSearchCV: 4 Ridge + 6 Elastic Net + 4 Random Forest + 8 boosting configurations; 22 configurations × 4 folds = 88 CV fits, plus refits.
- **13. Cross-validation — Executed:** Four expanding-window folds with 20 sequential one-week forecasts per fold. Median imputation, variance filtering and scaling are learned within each fold.
- **14. Evaluation — Executed:** Model selection by mean CV MAE. Locked-model holdout: MAE, RMSE, WAPE and R². All model holdout scores are descriptive; they do not change the selected winner.
- **15. Diagnostics — Executed:** Residual plot, average underprediction, lag-1 residual correlation and empirical interval coverage.
- **16. Interpretation — Executed:** Permutation importance on the calibration period; regularized linear model. Correlated features can share importance. SHAP/PDP/ICE were not run.
- **17. Business validation — Demonstrated; impact untested:** Compare forecast error to simple baselines. Use predicted volume and uncertainty for capacity planning. No measured revenue lift, incrementality, CAC reduction or ROI claim.
- **18. Deployment — Local scoring executed:** Serialized model and reusable score.py CLI tested. No cloud endpoint, Docker service, Salesforce writeback or production deployment.
- **19. Pipeline automation — Runnable; not scheduled:** run_pipeline.py reruns the workflow; score.py scores without retraining. Airflow/Prefect/CI scheduling is a proposed production step.
- **20. Monitoring — Offline snapshot executed:** Training/test feature means, standardized mean shifts and illustrative >1 standard-deviation review flags. No live alerts or latency monitoring.
- **21. Retraining — Plan only:** After new outcomes mature, evaluate a challenger with fresh time-based folds; recalibrate intervals and use a new holdout before promotion. Never repeatedly tune against this holdout.
- **22. Experimentation — Plan only:** For operational impact, randomize comparable teams or regions to forecast-informed staffing versus usual planning; predefine service-time/cost metrics. Budget causality needs a separate randomized experiment.